In [ ]:
## In this example, we will study output data frame from pandora.py configuration
#### 1. Opening each data frame and check structure
#### 2. Collect POT and scale factor to the target POT
#### 3. Merge evtdf and mcnudf for further study
#### 4. Draw some plots for each slice and for each pfp


import os
import sys
import lmfit
import numpy as np
import math
import uproot as uproot
import pickle
import pandas as pd
import gc

import matplotlib.pyplot as plt
import matplotlib.colors
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import ticker
from matplotlib.ticker import (AutoMinorLocator, MultipleLocator)
from matplotlib import gridspec

# Absolute path to cafpyana directory
print('Importing cafpyana utils...')
cafpyana_root = "/home/lpelegri/cafpyana"
# Add CAFpyana to the Python search path -- this will allow you to import cafpyana modules
sys.path.insert(0, cafpyana_root)

from analysis_village.unfolding.wienersvd import *
from analysis_village.unfolding.unfolding_inputs import *
from analysis_village.cc1pi.var_configs import *



from analysis_village.cc1pi.HelperFunctions import HelperFunctions
from analysis_village.cc1pi.Constants import CTE as CTE
from analysis_village.cc1pi.CutMasks import CutMasks
from analysis_village.cc1pi.CutMasks.MaskUtils import *
from analysis_village.cc1pi.DataFrameUtils import DFCleaning
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from analysis_village.cc1pi.GraphUtils.GraphsUtils import *
from analysis_village.cc1pi.GraphUtils.Utils import *

# import this repo's classes
import pyanalib.pandas_helpers as ph
import pyanalib.split_df_helpers as splh
import pyanalib.stat_helpers as sh

from pyanalib.split_df_helpers import *
from analysis_village.cc1pi.systematics.final_variable_configs import VariableConfig
from analysis_village.cc1pi.systematics.utils import *
from analysis_village.cc1pi.systematics.constants import *
from pyanalib.covariance import *
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *

np.seterr(divide='ignore', invalid='ignore', over='ignore')

# Load DataFrame MC

In [ ]:
from cols_to_keep import *
keys2load = ["cc1pi", "hdr", "histpotdf"] ## keys from the configuration file
bnb_path = "/exp/sbnd/data/users/lpelegri/cafpyana_data/mc_ar23p_pruned_for_data_mc_comp.df"
mc_bnb_df = load_df(bnb_path, keys2load, 100, filter_df = False, reprocess_df = False, reprocess_truth = False)
mc_bnb_evt_df = mc_bnb_df['cc1pi']
#mc_bnb_nu_df = mc_bnb_df['nudf']
mc_bnb_hdr_df = mc_bnb_df['hdr']
cols_to_keep = truth_cols_to_keep_slim
#mc_bnb_nu_df = mc_bnb_nu_df[cols_to_keep]

del mc_bnb_df
gc.collect()

keys2load = ["cc1pi", "hdr", "histpotdf"] ## keys from the configuration file
#load in time cosmic df
mc_in_time_cosmics_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_5e18_in_time_cosmics.df", keys2load, 100)
mc_in_time_cosmics_evt_df = mc_in_time_cosmics_df['cc1pi']
mc_in_time_cosmics_hdr_df = mc_in_time_cosmics_df['hdr']
del mc_in_time_cosmics_df
gc.collect()


#Load CV lowE dataframe
keys2load = ["cc1pi", "hdr", "histpotdf", "nudf"] ## keys from the configuration file
mc_bnb_lowE_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_1e20_lowE_CV.df", keys2load, 5, filter_df = False)
mc_bnb_lowE_evt_df = mc_bnb_lowE_df['cc1pi']
mc_bnb_lowE_nu_df = mc_bnb_lowE_df['nudf']
mc_bnb_lowE_hdr_df = mc_bnb_lowE_df['hdr']
del mc_bnb_lowE_df
gc.collect()

In [ ]:

#mc_bnb_lowE_nu_df = mc_bnb_lowE_nu_df[cols_to_keep]

In [ ]:
print(mc_bnb_lowE_nu_df.columns)

In [ ]:
#Load data
keys2load = ["cc1pi_good", "hdr", "histpotdf"] ## keys from the configuration file
data_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_data_fixdev_bnblight_quality_cut.df", keys2load, 100)
data_evt_df = data_df['cc1pi_good']
data_hdr_df = data_df['hdr']
del data_df
gc.collect()

keys2load = ["cc1pi", "hdr", "histpotdf"] ## keys from the configuration file
#load off beam light df
off_beam_light_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_data_offbeamlight.df", keys2load, 100)
off_beam_light_evt_df = off_beam_light_df['cc1pi']
off_beam_light_hdr_df = off_beam_light_df['hdr']
del off_beam_light_df
gc.collect()

In [ ]:
pot_weight_col = ('slc', 'wgt', '', '', '', '')
# BNB data

'''
data_tot_pot = data_hdr_df['pot'].sum()
data_gates = data_hdr_df.nbnbinfo.sum()
'''
print("data_tot_pot: %.3e" %(data_tot_pot))
print("data tot gates : %.3e" %(data_gates))
data_evt_df[pot_weight_col] = np.ones(len(data_evt_df))


# BNB MC
mc_tot_pot = mc_bnb_hdr_df['pot'].sum()
print("mc_tot_pot: %.3e" %(mc_tot_pot))
mc_pot_scale = data_tot_pot / mc_tot_pot
print("mc_pot_scale: %.3e" %(mc_pot_scale))
mc_bnb_evt_df[pot_weight_col] = mc_pot_scale * np.ones(len(mc_bnb_evt_df))

#Low E
mc_bnb_lowE_tot_pot = mc_bnb_lowE_hdr_df['pot'].sum()
print("dirt_tot_pot: %.3e" %(mc_bnb_lowE_tot_pot))
mc_bnb_lowE_pot_scale = data_tot_pot / mc_bnb_lowE_tot_pot
print("dirt_pot_scale: %.3e" %(mc_bnb_lowE_pot_scale))
mc_bnb_lowE_evt_df[pot_weight_col] = mc_bnb_lowE_pot_scale * np.ones(len(mc_bnb_lowE_evt_df))

intime_gates = mc_in_time_cosmics_hdr_df[mc_in_time_cosmics_hdr_df['first_in_subrun'] == 1]['ngenevt'].sum()
print("intime cosmics mc gates: {:.2e}".format(intime_gates))
f = 0.075
scale_intime_to_lightdata = (1-f)*data_gates/intime_gates
print("goal scale: {:.2f}".format(scale_intime_to_lightdata))
mc_in_time_cosmics_evt_df[pot_weight_col] = scale_intime_to_lightdata * np.ones(len(mc_in_time_cosmics_evt_df))

off_beam_data_gates = off_beam_light_hdr_df.noffbeambnb.sum()
print("intime cosmics data gates: {:.2e}".format(off_beam_data_gates))
f = 0.075
scale_off_beam_to_lightdata = (1-f)*data_gates/off_beam_data_gates
print("goal scale: {:.2f}".format(scale_off_beam_to_lightdata))
off_beam_light_evt_df[pot_weight_col] = scale_off_beam_to_lightdata * np.ones(len(off_beam_light_evt_df))

In [ ]:

mc_evt_df = mc_bnb_evt_df
del mc_bnb_evt_df  # free original immediately — mc_evt_df holds the reference now
gc.collect()

if "ar23p" in bnb_path:
    print("NOT MATCHING")
else:
    mc_evt_df = perform_truth_matching(mc_evt_df, mc_bnb_nu_df)
    del mc_bnb_nu_df
gc.collect()
print("Finished loading")

print("Starting concat for cosmics")
mc_bnb_lowE_evt_df = concat_shift_first_index(mc_bnb_lowE_evt_df, mc_in_time_cosmics_evt_df)
gc.collect()

print("TM for BNB nu df starting")
mc_bnb_lowE_evt_df = perform_truth_matching(mc_bnb_lowE_evt_df, mc_bnb_lowE_nu_df)
del mc_bnb_lowE_nu_df
gc.collect()
print("TM for BNB nu df done")

mc_evt_df = concat_shift_first_index(mc_evt_df, mc_bnb_lowE_evt_df)
del mc_bnb_lowE_evt_df  # free after final concat
gc.collect()


In [ ]:

#Load Sytematic samples
syst_keys = ["SystVarsCV","wiremod_YZ","wiremod_XZ_thetaXW", "0xSCE","2xSCE","PMTGainFluct", "PMTHighNoise", "PMTLowEff", "ccalm", "ccalp", "alpham", "alphap", "rm", "rp", "betam", "betap"]
colors = ["black", "C0","C1","C2","C3","C4","C5","C6","C7","C8","C9","C10","C11","C12","C13","C14","C15","C16"]
labels    = ["CV", r"Wiremod Y-Z",r"Wiremod X$\theta_{xw}$", "0xSCE", "2xSCE","PMTGainFluct", "PMTHighNoise", "PMTLowEff", "ccalm", "ccalp", "alpham", "alphap", "rm", "rp", "betam", "betap"]

'''
syst_keys = ["SystVarsCV","wiremod_YZ"]
colors = ["black", "C0","C1","C2","C3","C4","C5","C6","C7","C8","C9","C10","C11","C12","C13","C14","C15","C16"]
labels    = ["CV", r"Wiremod Y-Z"]
'''
'''
syst_keys = ["SystVarsCV","wiremod_YZ","wiremod_XZ_thetaXW","PMTGainFluct", "PMTHighNoise", "ccalm", "ccalp", "alpham", "alphap", "rm", "rp", "betam", "betap"]
colors = ["black", "C0","C1","C2","C3","C4","C5","C6","C7","C8","C9","C10","C11","C12","C13","C14","C15","C16"]
labels    = ["CV", r"Wiremod Y-Z",r"Wiremod X$\theta_{xw}$","PMTGainFluct", "PMTHighNoise", "ccalm", "ccalp", "alpham", "alphap", "rm", "rp", "betam", "betap"]
'''

keys2load = ["cc1pi", "hdr"] ## keys from the configuration file
syst_evt_dfs = {}
for key in syst_keys:
    syst_df = load_df(f"/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_{key}.df",keys2load,4, filter_df = False)

    syst_tot_pot = syst_df['hdr']['pot'].sum()
    syst_pot_scale = data_tot_pot / syst_tot_pot
    syst_evt_dfs[key] = syst_df['cc1pi']
    syst_evt_dfs[key][pot_weight_col] = syst_pot_scale * np.ones(len(syst_df['cc1pi']))
    
    del syst_df
    gc.collect()

# Test background composition

In [ ]:
mc_cumulative_masks = build_event_cumulative_masks(mc_evt_df, sideband = "")
data_cumulative_masks = build_event_cumulative_masks(data_evt_df, sideband = "")
in_time_cosmics_cumulative_masks = build_event_cumulative_masks(mc_in_time_cosmics_evt_df, sideband = "")
off_beam_data_cumulative_masks = build_event_cumulative_masks(off_beam_light_evt_df, sideband = "")

In [ ]:

# Loop
for name in mc_cumulative_masks.keys():
    n_mc   = get_n_evt(mc_evt_df,   mc_cumulative_masks[name],   use_weight=True)
    n_data = get_n_evt(data_evt_df, data_cumulative_masks[name], use_weight=False)
    print(f"{name:<15} | {n_mc:<12.2f} | {n_data:<12}")
    

In [ ]:
'''
for name in mc_cumulative_masks.keys():
    n_mc   = get_n_evt(mc_evt_df,   mc_cumulative_masks[name] & (mc_evt_df.slc.cut_var.n_exiting_pfps == 0),   use_weight=True)
    n_data = get_n_evt(data_evt_df, data_cumulative_masks[name] & (data_evt_df.slc.cut_var.n_exiting_pfps == 0), use_weight=False)
    print(f"{name:<15} | {n_mc:<12.2f} | {n_data:<12}")
'''

In [ ]:
for key in ["0p", "1p", "2plusp"]:
    mask = data_cumulative_masks["energy"] & mask_dict[key](data_evt_df)
    print(f"{key}: {get_n_evt(data_evt_df, use_weight=False, mask=mask)}")

In [ ]:
syst_masks = {}
for key, df in syst_evt_dfs.items():
    print(key)
    masks = build_event_cumulative_masks(df, sideband = "")
    syst_masks[key] = masks

In [ ]:
def get_evts(df, var_col, bins=None, verbose=True):
    var = df[var_col]
    if ('slc','wgt','','','','') in df.columns:
        weights = df.loc[:, ('slc','wgt','','','','')]
    else:
        if verbose:
            print("No pot_weight column found, returning 1 as weight (expected for data)")
        weights = np.ones_like(var)

    return var, weights

In [ ]:
def signal_hists(evtdf=None,
                 nudf=None,
                 var_config=None,
                 return_data=False,
                 plot=True,
                 textloc=[0.05, 0.55],
                 approval="internal",
                 save_fig=False,
                 save_name=None):

    bins     = var_config.bins
    reco_col = var_config.var_evt_reco_col
    get_fn   = get_clipped_evts if var_config.clip else get_evts

    # All selected — view, no copy
    var_allsel_reco, wgt_allsel_reco = get_fn(evtdf, reco_col, bins)

    # Signal mask — use loc to avoid copying the full df
    sig_mask = evtdf.truth.nu_categ == "CC1pi"  # boolean array only
    var_sel_reco  = var_allsel_reco[sig_mask]   # masked view into numpy array
    wgt_sel_reco  = wgt_allsel_reco[sig_mask]
    del sig_mask

    # Single-pass: compute sum_w and sum_w2 together
    nevts_allsel_reco,  _ = np.histogram(var_allsel_reco, weights=wgt_allsel_reco,    bins=bins)
    sum_w2_allsel_reco, _ = np.histogram(var_allsel_reco, weights=wgt_allsel_reco**2, bins=bins)
    del var_allsel_reco  # free before allocating signal histograms

    nevts_sel_reco,  _ = np.histogram(var_sel_reco, weights=wgt_sel_reco,    bins=bins)
    sum_w2_sel_reco, _ = np.histogram(var_sel_reco, weights=wgt_sel_reco**2, bins=bins)

    if return_data:
        return {
            "var_sel_reco":        var_sel_reco,
            "wgt_sel_reco":        wgt_sel_reco,
            "nevts_sel_reco":      nevts_sel_reco,
            "sum_w2_sel_reco":     sum_w2_sel_reco,
            "nevts_allsel_reco":   nevts_allsel_reco,
            "sum_w2_allsel_reco":  sum_w2_allsel_reco,
        }
    # if not return_data, all locals go out of scope here and are freed

In [ ]:
def get_univ_rates(cov_type="rate", 
                    evtdf=None, 
                    nudf=None, 
                    var_config=None, 
                    syst_name="", 
                    n_univ=100, 
                    bkgd_subtract=True,
                    plot=False):
    """
    for the GENIE uncertainty on the xsec measurement
    """

    if cov_type == "xsec":
        print("getting {} universes for {} uncertainty on the xsec".format(n_univ, syst_name))
        print(f"x-sec UNIT:{XSEC_UNIT}")
        scale_factor = XSEC_UNIT
    elif cov_type == "rate":
        print("getting {} universes for {} uncertainty on the event rate".format(n_univ, syst_name))
        scale_factor = 1.0
    else:
        raise ValueError("Invalid covariance type: {}, choose in [xsec, rate]".format(cov_type))

    bins = var_config.bins

    evtdf_signal = evtdf[evtdf.truth.nu_categ == "CC1pi"]
    # reco variable histogram, topology breakdown
    evtdf_div_topo = [evtdf[evtdf.truth.nu_categ == mode]for mode in topology_list]

    ret = signal_hists(evtdf, nudf, var_config, return_data=True, plot=plot)
    
    univ_events = []
    univ_effs   = []
    univ_smears = []

    for uidx in range(n_univ):
        syst_column = ("truth",syst_name,"univ_{}".format(uidx),"","","")

        
        current_evt_univ_weights = evtdf_signal[syst_column].to_numpy().copy()
        current_evt_univ_weights[np.isnan(current_evt_univ_weights)] = 1.0
        current_evt_univ_weights[current_evt_univ_weights >= 30] = 1.0
        
        signal_univ, _ = np.histogram(ret["var_sel_reco"], 
                                          weights=ret["wgt_sel_reco"]*current_evt_univ_weights,
                                          bins=bins)
        
        # TODO: this isn't computationally efficient, but it's useful for debugging
        # ---- uncertainty on the background rate ----
        # loop over background categories
        # + univ background - cv background
        # note: cv background subtraction cancels out with the cv background subtraction for the cv event rate. 
        #       doing it anyways for the plot of universes on background subtracted event rate.
   
        for this_evtdf in evtdf_div_topo[1:]:
            if var_config.clip:
                var, wgt = get_clipped_evts(this_evtdf, var_config.var_evt_reco_col, bins)
            else:
                var, wgt = get_evts(this_evtdf, var_config.var_evt_reco_col, bins)
                
            univ_wgt = this_evtdf[syst_column].copy()
            univ_wgt[np.isnan(univ_wgt)] = 1 ## IMPORTANT: make nan univ_wgt to 1. to ignore them
            univ_wgt[univ_wgt >= 30] = 1.0
            background_cv, _   = np.histogram(var, bins=bins, weights=wgt)
            background_univ, _ = np.histogram(var, bins=bins, weights=wgt*univ_wgt)

            if bkgd_subtract:
                signal_univ += (background_univ - background_cv)
            else:
                signal_univ += background_univ


        signal_univ *= scale_factor
        univ_events.append(signal_univ)

    univ_events = np.array(univ_events)

    if bkgd_subtract:
        cv_events = ret["nevts_sel_reco"]
        cv_events *= scale_factor
    else:
        cv_events = ret["nevts_allsel_reco"]
        cv_events *= scale_factor 

    return univ_events, cv_events


In [ ]:
# ==== fractional uncertainty plot ====
def plot_frac_unc(frac_unc_named_list,  # Expecting [(unc_array, "name"), ...]
                  var_config, 
                  plot_labels=["", "", ""],
                  approval="internal",
                  plot=True,
                  save_fig=False, 
                  save_name=None,
                  fig_ext=".pdf"):

    fig, ax = plt.subplots(figsize=(8, 6))
    
    max_val = 0
    for fidx, (frac_unc, label_name) in enumerate(frac_unc_named_list):
        color = "C{}".format(fidx)
        if len(frac_unc_named_list) == 1:
            color = "black"
        if label_name == "Total":
            color = "black"
            
        # Plotting in percent [%]
        ax.hist(var_config.bin_centers, bins=var_config.bins, 
                weights=frac_unc * 100, histtype="step", 
                color=color, linewidth=2, label=label_name)
        
        # Track max (account for the *100 scaling)
        current_max = np.nanmax(np.nan_to_num(frac_unc * 100, nan=0, posinf=0))
        if current_max > max_val:
            max_val = current_max

    # --- Axis Formatting ---
    ax.set_xlim(var_config.bins[0], var_config.bins[-1])
    ax.set_ylabel("Fractional Uncertainty [%]")
    
    if max_val == 0: max_val = 10.0 # Default to 10% if empty
    ax.set_ylim(0, max_val * 1.2) # Use 1.2 to leave room for the legend
    
    ax.set_title(plot_labels[2])
    ax.grid(True, linestyle='--', alpha=0.6)
    
    # Add the legend!
    ax.legend(loc='upper right', frameon=True)

    add_approval_text(approval, 0.02, 0.98, "left")
               
    if save_fig and save_name:
        plt.savefig(f"{save_name}{fig_ext}", bbox_inches='tight')

    if plot:
        plt.show()
    else:
        plt.close(fig)

In [ ]:
# AFTER: vectorised, safe division, same formula
def get_covariance_matrix(univ_events, cv_events):
    n_univ, n_bins = univ_events.shape

    # delta[u, i] = univ_events[u,i] - cv_events[i]
    delta = univ_events - cv_events[np.newaxis, :]  # (n_univ, n_bins)

    # Absolute covariance: mean over universes of outer product of deltas
    cov = (delta.T @ delta) / n_univ               # (n_bins, n_bins)

    # Fractional: divide each delta by cv before outer product
    # Guard against zero CV bins
    with np.errstate(divide='ignore', invalid='ignore'):
        cv_safe   = np.where(cv_events != 0, cv_events, np.inf)
        delta_frac = delta / cv_safe[np.newaxis, :]  # (n_univ, n_bins)

    cov_frac = (delta_frac.T @ delta_frac) / n_univ  # (n_bins, n_bins)

    cov      = np.nan_to_num(cov,      nan=0.0)
    cov_frac = np.nan_to_num(cov_frac, nan=0.0)
    cov_frac = np.clip(cov_frac, -25.0, 25.0)

    # Correlation matrix
    diag     = np.sqrt(np.diag(cov))
    denom    = np.outer(diag, diag)
    with np.errstate(divide='ignore', invalid='ignore'):
        corr = np.where(denom != 0, cov / denom, 0.0)

    return {"cov_frac": cov_frac, "cov": cov, "corr": corr}

In [ ]:
'''
def variation_hists(evtdfs=None, var_name=None, clip = False, 
                    nevts_list=None,
                    datadf=None,
                    bins=None,
                    var_colors=None, var_labels=None,
                    plot_labels=["", "", ""],
                    vline = None,
                    textloc=[0.05, 0.55],
                    approval="internal",
                    plot=True,
                    save_fig=False, save_name=None): 

    bin_centers = 0.5 * (bins[:-1] + bins[1:])

    if evtdfs is not None:
        n_vars = len(evtdfs)
    elif nevts_list is not None:
        n_vars = len(nevts_list)
    else:
        raise ValueError("Either evtdfs or nevts_list must be provided")

    # get distribution from dfs
    if evtdfs is not None:
        vardfs, wgtdfs = [], []
        nevts_list = []
        mc_stat_err_list = []
        for df in evtdfs:
            if clip:
                vardf, wgtdf = get_clipped_evts(df, var_name, bins)
            else:
                vardf, wgtdf = get_evts(df, var_name, bins)
                
           
            vardfs.append(vardf)
            wgtdfs.append(wgtdf)


        # for sidx in range(n_vars):
            nevts, _ = np.histogram(vardf, bins=bins, weights=wgtdf)
            total_mc_err2, _ = np.histogram(vardf, bins=bins, weights=wgtdf**2)
            mc_stat_err = np.sqrt(total_mc_err2)
            nevts_list.append(nevts)
            mc_stat_err_list.append(mc_stat_err)

    return nevts_list
'''
def variation_hists(evtdfs=None, var_name=None, clip=False,
                    nevts_list=None,
                    datadf=None,
                    bins=None,
                    var_colors=None, var_labels=None,
                    plot_labels=["", "", ""],
                    vline=None,
                    textloc=[0.05, 0.55],
                    approval="internal",
                    plot=True,
                    save_fig=False, save_name=None):
    bin_centers = 0.5 * (bins[:-1] + bins[1:])

    if evtdfs is None and nevts_list is None:
        raise ValueError("Either evtdfs or nevts_list must be provided")

    if evtdfs is not None:
        nevts_list = []
        mc_stat_err_list = []

        for df in evtdfs:  # works with both list and generator
            if clip:
                vardf, wgtdf = get_clipped_evts(df, var_name, bins)
            else:
                vardf, wgtdf = get_evts(df, var_name, bins)

            nevts, _       = np.histogram(vardf, bins=bins, weights=wgtdf)
            total_mc_err2, _ = np.histogram(vardf, bins=bins, weights=wgtdf**2)

            nevts_list.append(nevts)
            mc_stat_err_list.append(np.sqrt(total_mc_err2))
            del vardf, wgtdf, total_mc_err2  # free per-df arrays immediately

        n_vars = len(nevts_list)  # now safe — list of small histograms only
    else:
        n_vars = len(nevts_list)

    return nevts_list

In [ ]:
from functools import partial

unisim_keys = []
paired_syst = {} 

detvar_plotter = partial(
    variation_hists,
    var_colors=colors,
    var_labels=labels,
    plot = True
)

# Identify Unisim (1 univ) vs Paired/Multisim (2 univ)
for key in syst_keys:
    if key == "SystVarsCV": continue
    if key.endswith('m') or key.endswith('p'):
        base = key[:-1]
        if base not in paired_syst:
            paired_syst[base] = [None, None]
        if key.endswith('m'): paired_syst[base][0] = key
        else: paired_syst[base][1] = key
    else:
        unisim_keys.append(key)

# Version that gets everything for the Data/MC comparison

In [ ]:
def filter_df(df, extra_mask, cut_mask, config):
    """Applies cut masks without copying — returns a view where possible."""
    mask = cut_mask & mask_dict[extra_mask](df)
    filtered = df.loc[mask]
    if config.first_per_slice:
        # groupby.first() must allocate, unavoidable
        return filtered.groupby(level=SLICE_LEVELS, sort=False).first()
    return filtered  # view, no copy

In [ ]:
mask_dict = {
    "none": lambda df: pd.Series(True, index=df.index),
    "pandora_primary": lambda df: CutMasks.is_pandora_primary_mask(df),
    "analysis_primary": lambda df: CutMasks.is_analysis_primary_mask(df),
    "primary_track": lambda df: CutMasks.is_primary_track_mask(df),
    "primary_shower": lambda df: CutMasks.is_primary_shower_mask(df),
    "MIP_candidate": lambda df: CutMasks.is_MIP_candidate_mask(df),
    "contained_MIP_candidate": lambda df: CutMasks.is_MIP_candidate_mask(df) & (df.pfp.is_exiting == False),
    "exiting_MIP_candidate": lambda df: CutMasks.is_MIP_candidate_mask(df) & (df.pfp.is_exiting == True),
    "muon_exiting": lambda df: df.slc.measure_var.muon_contained == False,
    "muon_contained": lambda df: df.slc.measure_var.muon_contained == True,
    "0p": lambda df: df.slc.measure_var.num_protons == 0,
    "1p": lambda df: df.slc.measure_var.num_protons == 1,
    "2plusp": lambda df: df.slc.measure_var.num_protons > 1,
}


In [ ]:
gc.collect()

In [ ]:
cuts = ["cosmic","t0","FV","nu_score","track","chi2","shower","angle","proton_BDT","energy"]

config_angle_between_candidates = FullHistogramConfig(
    file_name = "angle_between_candidates",      
    var_evt_reco_col=('slc','measure_var','angle_between_candidates','','',''),
    truth_column = nu_categ_column,
    first_per_slice = True,
    start_cut = "shower",
    end_cut = "angle",
    bins=np.linspace(0, np.pi, 41),
    xlabel=r'Angle between MIP candidates [rad]',
    cut_value = [CTE.max_angle_between_candidates],
    ylabel=slices_y_label
)

config_analysis_primary_track_score = FullHistogramConfig(
    file_name = "analysis_primary_track_score", 
    var_evt_reco_col=('pfp','trackScore','','','',''),
    truth_column=pfp_truth_column,
    first_per_slice = False,
    start_cut = "nu_score",
    end_cut = "nu_score",
    extra_mask = "analysis_primary",
    bins=np.linspace(0, 1, 41),
    xlabel='analysis primary track score (vtx dist inc.)',
    ylabel= pfps_y_label,
    cut_value = [CTE.min_track_score],
    clip = False
)

config_contained_MIP_candidate_chi2_mu = FullHistogramConfig(
    file_name = "contained_MIP_candidate_chi2_mu", 
    var_evt_reco_col=('pfp','trk','chi2pid','best','chi2_muon',''),
    truth_column=pfp_truth_column,
    first_per_slice = False,
    start_cut = "angle",
    end_cut = "energy",
    extra_mask = "contained_MIP_candidate",
    bins=np.linspace(0, 20, 41),
    xlabel=r'contained MIP candidates $\chi^2_{\mu}$',
    ylabel= pfps_y_label,
    cut_value = [CTE.MIP_candidate_max_muon_score],
    clip = False
)

config_contained_MIP_candidate_chi2_p = FullHistogramConfig(
    file_name = "contained_MIP_candidate_chi2_p", 
    var_evt_reco_col=('pfp','trk','chi2pid','best','chi2_proton',''),
    truth_column=pfp_truth_column,
    first_per_slice = False,
    start_cut = "angle",
    end_cut = "energy",
    extra_mask = "contained_MIP_candidate",
    bins=np.linspace(0, 300, 41),
    xlabel=r'contained MIP candidates $\chi^2_{p}$',
    ylabel= pfps_y_label,
    cut_value = [CTE.MIP_candidate_min_proton_score],
    clip = False
)

config_contained_MIP_candidate_scatter_angle =  FullHistogramConfig(
    file_name = "contained_MIP_candidate_scatter_angle", 
    var_evt_reco_col=('pfp','scatter_angle_ratio','','','',''),
    truth_column=pfp_truth_column,
    first_per_slice = False,
    start_cut = "energy",
    end_cut = "energy",
    extra_mask = "contained_MIP_candidate",
    bins=np.linspace(0, 1, 41),
    xlabel=r'Contained MIP candidates $\frac{MCS max scatter}{MCS total scatter}$',
    ylabel= pfps_y_label
)

config_contained_MIP_candidate_max_daughter_hits = FullHistogramConfig(
    file_name = "contained_MIP_candidate_max_daughter_hits", 
    var_evt_reco_col=('pfp','max_daughter_hits','','','',''),
    truth_column=pfp_truth_column,
    first_per_slice = False,
    start_cut = "energy",
    end_cut = "energy",
    extra_mask = "contained_MIP_candidate",
    bins=np.linspace(10, 300, 41),
    xlabel=r'Contained MIP candidates daughter max hits',
    ylabel= pfps_y_label
)

config_contained_MIP_candidate_frac50 = FullHistogramConfig(
    file_name = "contained_MIP_candidate_frac50", 
    var_evt_reco_col=('pfp','trk','frac50','','',''),
    truth_column=pfp_truth_column,
    first_per_slice = False,
    start_cut = "angle",
    end_cut = "angle",
    extra_mask = "contained_MIP_candidate",
    bins=np.linspace(0, 1, 41),
    xlabel=r'Contained MIP candidates RR frac. with 50% energy',
    ylabel= pfps_y_label
)

config_contained_MIP_candidate_chi2_exp_pol = FullHistogramConfig(
    file_name = "contained_MIP_candidate_chi2_exp_pol", 
    var_evt_reco_col=('pfp','trk','chi2_exp_pol','','',''),
    truth_column=pfp_truth_column,
    first_per_slice = False,
    start_cut = "angle",
    end_cut = "energy",
    extra_mask = "contained_MIP_candidate",
    bins=np.linspace(0, 1, 41),
    xlabel=r'Contained MIP candidates $\chi^2_{pol_0} / \chi^2_{exp}$',
    ylabel= pfps_y_label
)

config_contained_MIP_candidate_bdt_score_proton = FullHistogramConfig(
    file_name = "contained_MIP_candidate_bdt_score_proton", 
    var_evt_reco_col=('pfp','trk','bdt_proton_score','','',''),
    truth_column=pfp_truth_column,
    first_per_slice = False,
    start_cut = "angle",
    end_cut = "angle",
    extra_mask = "contained_MIP_candidate",
    bins=np.linspace(-5, 9, 41),
    xlabel=r'Contained MIP candidates proton BDT score',
    cut_value = [CTE.BDT_proton_max_score],
    ylabel= pfps_y_label
)

config_contained_MIP_candidate_bdt_score_muon_pion = FullHistogramConfig(
    file_name = "contained_MIP_candidate_bdt_score_muon_pion", 
    var_evt_reco_col=('pfp','trk','bdt_muon_pion_score','','',''),
    truth_column=pfp_truth_column,
    first_per_slice = False,
    start_cut = "energy",
    end_cut = "energy",
    extra_mask = "contained_MIP_candidate",
    bins=np.linspace(-5, 7, 41),
    xlabel=r'Contained MIP candidates $\mu/\pi$ separation BDT score',
    ylabel= pfps_y_label
)


measure_var_vec = [
    config_angle_between_candidates,
]

particle_vars_vec = [
    config_contained_MIP_candidate_bdt_score_muon_pion,
    config_pandora_primary_vtx_distance,
    config_analysis_primary_track_score ,
    config_analysis_primary_len,

    config_primary_track_chi2_mu,
    config_primary_track_chi2_p,
    config_contained_MIP_candidate_chi2_mu,
    config_contained_MIP_candidate_chi2_p,
    config_contained_MIP_candidate_scatter_angle, 
    config_contained_MIP_candidate_frac50,
    config_contained_MIP_candidate_chi2_exp_pol,
    config_contained_MIP_candidate_bdt_score_proton,
]



In [ ]:

def get_stat_covariance_matrix(cv_contents, sum_w2):
    """
    cv_contents: array of bin contents (sum of weights)
    sum_w2: array of the sum of the squares of the weights per bin
    """
    cv_contents = np.asarray(cv_contents)
    sum_w2 = np.asarray(sum_w2)
    n_bins = len(cv_contents)

    # 1. Variance for weighted Poisson is Sum(W^2)
    cov = np.diag(sum_w2)

    # 2. Fractional covariance: Var / (Content^2) = Sum(W^2) / (Sum W)^2
    with np.errstate(divide='ignore', invalid='ignore'):
        # This is the squared fractional error
        frac_variance = np.where(cv_contents > 0, sum_w2 / (cv_contents**2), 0.0)
        cov_frac = np.diag(frac_variance)

    # 3. Correlation matrix
    corr = np.eye(n_bins)

    return {
        "cov": cov,
        "cov_frac": cov_frac,
        "corr": corr,
    }

In [ ]:
for name in mc_cumulative_masks.keys():
    n_mc   = get_n_evt(mc_evt_df,   mc_cumulative_masks[name],   use_weight=True)
    n_data = get_n_evt(data_evt_df, data_cumulative_masks[name], use_weight=False)
    print(f"{name:<15} | {n_mc:<12.2f} | {n_data:<12}")

In [ ]:
def plot_stacked_histogram_with_ratio(
    mc_df,
    data_df,
    config,
    cov_frac_matrix=None,
    cov_matrix=None,
    title: str = None,
    weight_column: tuple = None,
    data_pot: float = None,
    normalize: bool = False,
    show_stats: bool = True,
    symmetric_ratio: bool = False,
    divide_by_bin_width: bool = False
):
    # 1. Pre-processing and Scaling
    slice_levels = ['__ntuple', 'entry', 'rec.slc..index']
    if config.first_per_slice:
        mc_df   = mc_df.groupby(level=slice_levels, sort=False).first()
        data_df = data_df.groupby(level=slice_levels, sort=False).first()

    mc_data    = mc_df[config.var_evt_reco_col]
    mc_types   = mc_df[config.truth_column]
    mc_weights = mc_df[weight_column] if weight_column in mc_df.columns else pd.Series(1.0, index=mc_df.index)

    # --- Palette Selection ---
    present_categories = set(mc_types.dropna().unique())
    palettes = [category_colors, category_colors_pfp, proton_distinction_category_colors, genie_category_colors]
    chosen_map, max_overlap = category_colors, -1
    for p in palettes:
        overlap = len(present_categories.intersection(p.keys()))
        if overlap > max_overlap:
            max_overlap = overlap
            chosen_map  = p

    # 2. Sorting Categories
    category_totals = [(t, mc_weights[mc_types == t].sum()) for t in mc_types.dropna().unique()]
    signal_keys     = ["CC1pi"]
    signals     = sorted([x for x in category_totals if     x[0] in signal_keys], key=lambda x: x[1], reverse=True)
    backgrounds = sorted([x for x in category_totals if not x[0] in signal_keys], key=lambda x: x[1], reverse=True)

    sorted_types   = [x[0] for x in backgrounds + signals]
    stack_data_mc  = [mc_data[mc_types == t].dropna() for t in sorted_types]
    stack_weights  = [mc_weights[mc_types == t].loc[mc_data[mc_types == t].dropna().index] for t in sorted_types]

    # 3. Setup Figure
    fig      = plt.figure(figsize=(10, 8))
    gs       = gridspec.GridSpec(2, 1, height_ratios=[4, 1], hspace=0.07)
    ax_top   = fig.add_subplot(gs[0])
    ax_ratio = fig.add_subplot(gs[1], sharex=ax_top)

    # 4. TOP PLOT — clip if needed
    max_bin_edge      = config.bins[-1]
    stack_mc_clipped  = (
        [np.clip(d, config.bins[0], max_bin_edge) for d in stack_data_mc]
        if config.clip else stack_data_mc
    )

    colors         = [chosen_map.get(t, "#7f7f7f") for t in sorted_types]
    all_mc_weights = pd.concat(stack_weights)

    mc_sum, bins = np.histogram(
        pd.concat(stack_mc_clipped),
        bins=config.bins,
        weights=all_mc_weights
    )
    # NEW: Compute raw unweighted MC entries for the threshold guard
    mc_raw_counts, _ = np.histogram(
        pd.concat(stack_mc_clipped),
        bins=config.bins
    )
    
    bin_centers = (bins[:-1] + bins[1:]) / 2
    bin_widths  = np.diff(bins)

    # --- MC UNCERTAINTY ---
    if cov_frac_matrix is not None:
        frac_err = np.sqrt(np.diag(cov_frac_matrix))
        mc_error = frac_err * mc_sum
    else:
        mc_sum_w2, _ = np.histogram(
            pd.concat(stack_mc_clipped),
            bins=config.bins,
            weights=all_mc_weights ** 2
        )
        mc_error         = np.sqrt(mc_sum_w2)
        stat_cov         = get_stat_covariance_matrix(mc_sum, mc_sum_w2)
        cov_matrix       = stat_cov["cov"]
        cov_frac_matrix  = stat_cov["cov_frac"]

    if normalize:
        norm_fact = mc_sum.sum()
        if norm_fact > 0:
            mc_sum    /= norm_fact
            mc_error  /= norm_fact
            stack_weights = [w / norm_fact for w in stack_weights]

    # --- DATA ---
    cols_to_check = [config.var_evt_reco_col]
    if weight_column in data_df.columns:
        cols_to_check.append(weight_column)
    valid_df     = data_df.dropna(subset=cols_to_check)
    data         = valid_df[config.var_evt_reco_col]
    data_weights = valid_df[weight_column] if weight_column in valid_df.columns else np.ones(len(data))

    data_clipped = np.clip(data, config.bins[0], max_bin_edge) if config.clip else data

    data_counts, _ = np.histogram(data_clipped, bins=bins, weights=data_weights)
    data_sum_w2, _ = np.histogram(data_clipped, bins=bins, weights=data_weights ** 2)

    if normalize:
        denom            = len(data) if len(data) > 0 else 1
        data_plot_counts = data_counts / denom
        data_errors      = np.sqrt(data_sum_w2) / denom
    else:
        data_plot_counts = data_counts.astype(float)
        data_errors      = np.sqrt(data_sum_w2)

    # 6. STATISTICS
    # --- UPDATED CHI2 LOGIC ---
    # We add the real Data Poisson statistical error (np.diag(data_counts)) to the covariance matrix.
    # We pass data_counts and mc_sum explicitly prior to executing any bin-width operations.
    chi2_val, ndof, p_val = get_chi2(
        data=data_counts, 
        mc=mc_sum, 
        cov=cov_matrix + np.diag(data_counts), 
        mc_raw=mc_raw_counts
    )
    
    if divide_by_bin_width:
        mc_sum           /= bin_widths
        mc_error         /= bin_widths
        data_plot_counts /= bin_widths
        data_errors      /= bin_widths
        new_stack_weights = []
        for i, d in enumerate(stack_mc_clipped):
            bin_indices = np.clip(np.digitize(d, bins) - 1, 0, len(bin_widths) - 1)
            new_stack_weights.append(stack_weights[i] / bin_widths[bin_indices])
        stack_weights = new_stack_weights

    ax_top.hist(stack_mc_clipped, bins=bins, stacked=True, weights=stack_weights,
                histtype='stepfilled', color=colors, alpha=0.3)
    ax_top.hist(stack_mc_clipped, bins=bins, stacked=True, weights=stack_weights,
                histtype='step', color=colors, linewidth=2)
    ax_top.bar(bin_centers, 2 * mc_error, bottom=mc_sum - mc_error, width=bin_widths,
               edgecolor='grey', facecolor='grey', alpha=0.2, linewidth=0)
    ax_top.bar(bin_centers, 2 * mc_error, bottom=mc_sum - mc_error, width=bin_widths,
               edgecolor='grey', facecolor='none', hatch='////', alpha=0.5, linewidth=0)
    ax_top.errorbar(bin_centers, data_plot_counts, yerr=data_errors, xerr=bin_widths / 2,
                    fmt='ko', markersize=6, zorder=10, capsize=0)

    # 5. RATIO PLOT
    with np.errstate(divide='ignore', invalid='ignore'):
        ratio       = np.divide(data_plot_counts, mc_sum, out=np.zeros_like(data_plot_counts), where=mc_sum != 0)
        ratio_error = np.divide(data_errors,       mc_sum, out=np.zeros_like(data_errors),       where=mc_sum != 0)
        mc_rel_error = np.divide(mc_error,          mc_sum, out=np.zeros_like(mc_error),            where=mc_sum != 0)

    valid_ratio = mc_sum > 0
    if np.any(valid_ratio) and symmetric_ratio:
        data_extrema  = np.maximum(
            np.abs((ratio[valid_ratio] + ratio_error[valid_ratio]) - 1),
            np.abs((ratio[valid_ratio] - ratio_error[valid_ratio]) - 1)
        )
        mc_extrema    = mc_rel_error[valid_ratio]
        max_deviation = np.max(np.maximum(data_extrema, mc_extrema))
        y_padding     = max(max_deviation * 1.4, 0.1)
        ax_ratio.set_ylim(1 - y_padding, 1 + y_padding)
    elif np.any(valid_ratio):
        max_deviation = np.max(np.abs(ratio[valid_ratio] - 1) + ratio_error[valid_ratio])
        y_padding     = max(min(max_deviation * 1.4, 0.75), 0.1)
        ax_ratio.set_ylim(1 - y_padding, 1 + y_padding)
    else:
        ax_ratio.set_ylim(0.5, 1.5)

    ax_ratio.bar(bin_centers, 2 * mc_rel_error, bottom=1 - mc_rel_error, width=bin_widths,
                 edgecolor='grey', facecolor='grey', alpha=0.2, linewidth=0)
    ax_ratio.bar(bin_centers, 2 * mc_rel_error, bottom=1 - mc_rel_error, width=bin_widths,
                 edgecolor='grey', facecolor='none', hatch='////', alpha=0.4, linewidth=0)
    ax_ratio.errorbar(bin_centers, ratio, yerr=ratio_error, xerr=bin_widths / 2,
                      fmt='ko', markersize=6, capsize=0)
    ax_ratio.axhline(1.0, color='#d62728', linestyle='--', linewidth=2)

    # 5.5 VERTICAL CUT LINE
    cut_val  = getattr(config, 'cut_value', None)
    draw_cut = False
    if cut_val is not None:
        draw_cut = (
            any(v != -999 for v in cut_val)
            if isinstance(cut_val, (list, np.ndarray))
            else (cut_val != -999)
        )

    cut_hand = Line2D([0], [0], color='black', linestyle='--', linewidth=2, label='Selection Cut')
    if draw_cut:
        cuts_to_draw = cut_val if isinstance(cut_val, (list, np.ndarray)) else [cut_val]
        for ax in [ax_top, ax_ratio]:
            for val in cuts_to_draw:
                if val != -999:
                    ax.axvline(val, color='black', linestyle='--', linewidth=2, zorder=2)

    # 7. LEGEND
    total_data_counts = len(data_clipped)
    mc_hand  = [Patch(facecolor=to_rgba(chosen_map.get(t, "#7f7f7f"), 0.3),
                      edgecolor=chosen_map.get(t, "#7f7f7f"),
                      label=bkg_name_nice_map.get(t, t)) for t in sorted_types]
    err_label = 'MC Total Error' if show_stats else 'MC Stat. Error'
    err_hand  = Patch(edgecolor='grey', facecolor='none', hatch='////', alpha=0.5, label=err_label)
    dat_hand  = Line2D([0], [0], color='black', marker='o', linestyle='', label='Data', markersize=8)

    all_handles = mc_hand + [err_hand, dat_hand]
    all_labels  = [h.get_label() for h in mc_hand] + [err_label, 'Data']

    if draw_cut:
        all_handles.append(cut_hand)
        all_labels.append(cut_hand.get_label())

    if show_stats:
        if ndof > 0 and chi2_val is not None:
            chi2_str = f"$\\chi^{{2}}$ / ndf: {chi2_val:.2f} / {ndof} = {chi2_val/ndof:.3f}"
        else:
            chi2_str = f"$\\chi^{{2}}$: N/A"
        p_value_str      = f"$p_{{\\mathrm{{value}}}}$ = {p_val:.3f}" if p_val is not None else "$p$: N/A"
        data_str         = f'$N_{{\\mathrm{{Data}}}}$ = {total_data_counts}'
        chi2_handle      = Patch(color='none', label=chi2_str)
        p_value_handle   = Patch(color='none', label=p_value_str)
        N_data_evts_handle = Patch(color='none', label=data_str)
        all_handles += [chi2_handle, p_value_handle, N_data_evts_handle]
        all_labels  += [chi2_str, p_value_str, data_str]

    n_cols = (len(all_handles) + 3) // 4
    leg = ax_top.legend(
        handles=all_handles, labels=all_labels,
        loc='upper ' + config.stats_horizontal_alignment, ncol=n_cols,
        fontsize=12, framealpha=1.0, edgecolor='black', fancybox=False,
        borderaxespad=1, columnspacing=1.5, handlelength=1.5, handletextpad=0.5,
    )
    leg.get_frame().set_linewidth(1.5)

    plt.draw()
    if show_stats:
        texts = leg.get_texts()
        for t in texts[-3:]:
            t.set_position((-28, 0))

    # 8. FINAL STYLING
    ax_top.set_xlim(bins[0], bins[-1])
    ax_top.set_ylim(0, ax_top.get_ylim()[1] * 1.4)

    ylabel = config.ylabel
    if divide_by_bin_width:
        ylabel += " / bin width"

    ax_top.set_ylabel(f"{ylabel} (POT = {data_pot:.2e})")
    ax_ratio.set_ylabel("Data/MC")
    ax_ratio.set_xlabel(config.xlabel, fontsize=20)
    ax_ratio.tick_params(axis='x', which='both', direction='inout', length=6)
    ax_top.set_title("")

    plt.setp(ax_top.get_xticklabels(), visible=False)
    fig.align_ylabels([ax_top, ax_ratio])
    plt.subplots_adjust(top=0.92, bottom=0.12, left=0.12, right=0.95, hspace=0.07)

    plt.show()
    return fig, p_val

In [ ]:
import numpy as np
from scipy.stats import chi2 as chi2_dist

def get_chi2(data, mc, cov, mc_raw=None, n_params=0):
    data = np.atleast_1d(data)
    mc   = np.atleast_1d(mc)
    
    # Default to scaled MC if raw MC isn't provided explicitly
    if mc_raw is None:
        mc_raw = mc
    else:
        mc_raw = np.atleast_1d(mc_raw)

    if cov.shape != (len(data), len(data)):
        raise ValueError(f"Covariance shape {cov.shape} doesn't match data length {len(data)}")

    # Masking logic: 
    # 1. Elements must be valid numbers (not NaN or Infinite)
    # 2. Observed Data entries must be >= 10
    # 3. Unweighted, raw simulated MC events must be >= 10
    stat_threshold = 10
    mask = (
        np.isfinite(data) & 
        np.isfinite(mc) & 
        (data >= stat_threshold) & 
        (mc_raw >= stat_threshold)
    )

    data_f = data[mask]
    mc_f   = mc[mask]
    cov_f  = cov[np.ix_(mask, mask)]

    n_bins = mask.sum()
    ndof   = n_bins - n_params

    if ndof <= 0:
        return np.nan, ndof, np.nan

    # A very high condition number means the matrix is singular/unstable.
    if np.linalg.cond(cov_f) > 1e12:
        return np.nan, ndof, np.nan

    # Standard physics convention: Delta = Data - MC
    delta = data_f - mc_f
    
    try:
        # FIXED: Try np.linalg.solve first! It handles stiff, correlated 
        # systematic matrices with significantly better numerical stability.
        chi2_val = delta @ np.linalg.solve(cov_f, delta)
    except np.linalg.LinAlgError:
        try:
            # Fallback to standard explicit inversion only if solve hits a corner case
            chi2_val = delta @ np.linalg.inv(cov_f) @ delta
        except np.linalg.LinAlgError:
            return np.nan, ndof, np.nan

    # Protect against any unphysical negative chi2 values from floating-point precision limits
    if chi2_val < 0:
        chi2_val = 0.0

    pval = chi2_dist.sf(chi2_val, ndof)

    return chi2_val, ndof, pval

In [ ]:
import os, shutil, tarfile
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd


#config_vec = [config_bc_flash_score]  #+ TKI_genie_categ_vec 
#config_vec= [config_MIP_candidate_chi2_mu, config_contained_MIP_candidate_chi2_mu, config_exiting_MIP_candidate_chi2_mu]

config_vec =  particle_vars_vec + measure_var_vec +  n_pfps_config_vec + simple_vars_vec_no_flash_time #+ TKI_genie_categ_vec + TKI_vec + 

FULL_SYST = True


if FULL_SYST:
    parent_path = "/exp/sbnd/data/users/lpelegri/Graphs/MCDataCompGraphsFullErr"
else:
    parent_path = "/exp/sbnd/data/users/lpelegri/Graphs/MCDataCompGraphs"

'''
if plot_sideband:
    parent_path += "sideband"
'''

good_base_path, bad_base_path = os.path.join(parent_path, "good"), os.path.join(parent_path, "bad")
tar_output_path = f"{parent_path}.tar"

PVALUE_THRESHOLD = 0.05
PLOT_IND_UNCR = False
PLOT_GENIE_CATEG = True
divide_by_bin_width = False
GENIE_TAGS = [False]
WGT_COL = ('slc', 'wgt', '', '', '', '')
SLICE_LEVELS = ['__ntuple', 'entry', 'rec.slc..index']

# Color codes
RED, RESET = "\033[31m", "\033[0m"

# Clean workspace
'''
for path in [parent_path, tar_output_path]:
    if os.path.exists(path):
        shutil.rmtree(path) if os.path.isdir(path) else os.remove(path)
os.makedirs(good_base_path, exist_ok=True); os.makedirs(bad_base_path, exist_ok=True)
'''

# --- Main Analysis Loop ---
smaller_binning_cuts = ["containment","michel","extra_pion","energy"]

# --- Main Analysis Loop ---
for config in config_vec:
    idx_range = sorted([cuts.index(config.start_cut), cuts.index(config.end_cut)])
    selected_cuts = cuts[idx_range[0] : idx_range[1] + 1]

    for cut in selected_cuts:
        print(f"--- Processing Cut: {cut} ---")
        if cut in smaller_binning_cuts:
            config.bins = np.linspace(config.bins[0], config.bins[-1], 31)
            # Recompute bin centers so the array shape matches the new binning!
            config.bin_centers = 0.5 * (config.bins[:-1] + config.bins[1:])
        
        # 1. Prepare DataFrames
        curr_mc   = filter_df(mc_evt_df,   config.extra_mask, mc_cumulative_masks[cut],   config)
        curr_data = filter_df(data_evt_df, config.extra_mask, data_cumulative_masks[cut], config)
        
        # 2. Base Statistics
        ret       = signal_hists(evtdf=curr_mc, var_config=config, return_data=True, plot=False)
        cv_events = ret["nevts_allsel_reco"]
        ret_stats = get_stat_covariance_matrix(cv_events, ret["sum_w2_allsel_reco"])
        del ret

        total_cov      = ret_stats["cov"]       # we own this, no copy needed
        total_cov_frac = ret_stats["cov_frac"]
        frac_unc_list  = [(np.sqrt(np.diag(total_cov_frac)), "MC Stat")]
        del ret_stats

        # 3. Systematics
        if FULL_SYST:

            # A. Detector Systematics
            print("--- Det Syst ---")

            # Generator: one filtered syst df in memory at a time
            syst_dfs = (
                filter_df(syst_evt_dfs[k], config.extra_mask, syst_masks[k][cut], config)
                for k in syst_keys
            )
            n_det = detvar_plotter(
                evtdfs=syst_dfs,
                var_name=config.var_evt_reco_col,
                clip=config.clip,
                bins=config.bins
            )
            # n_det is a list of small histograms — generator is now exhausted and freed

            key_to_idx    = {key: i for i, key in enumerate(syst_keys)}
            cv_events_det = n_det[0]
            det_cov_frac  = np.zeros_like(total_cov_frac)
            det_cov  = np.zeros_like(total_cov)
            uncertanties_list = []

            for syst_key in unisim_keys:
                if syst_key not in key_to_idx:
                    continue
                res = get_covariance_matrix(np.array([n_det[key_to_idx[syst_key]]]), cv_events_det)
                uncertanties_list.append((np.sqrt(np.diag(res["cov_frac"])), syst_key))
                det_cov_frac += res["cov_frac"]
                det_cov += res["cov"]

            for base_name, keys in paired_syst.items():
                m_idx = key_to_idx.get(keys[0])
                p_idx = key_to_idx.get(keys[1])
                if m_idx is None or p_idx is None:
                    continue
                res = get_covariance_matrix(np.array([n_det[m_idx], n_det[p_idx]]), cv_events_det)
                uncertanties_list.append((np.sqrt(np.diag(res["cov_frac"])), base_name))
                det_cov_frac += res["cov_frac"]
                det_cov += res["cov"]

            del n_det, key_to_idx, cv_events_det

            if PLOT_IND_UNCR:
                plot_frac_unc(uncertanties_list + [(np.sqrt(np.diag(det_cov_frac)), "total")], config)

            # B. Cosmics
            #print("--- Cosmics Syst ---")
            cos_mask = mc_cumulative_masks[cut]    & mask_dict[config.extra_mask](mc_in_time_cosmics_evt_df)
            off_mask = off_beam_data_cumulative_masks[cut] & mask_dict[config.extra_mask](off_beam_light_evt_df)

            d_cnts, _ = np.histogram(
                off_beam_light_evt_df.loc[off_mask, config.var_evt_reco_col],
                bins=config.bins,
                weights=off_beam_light_evt_df.loc[off_mask, WGT_COL]
            )
            m_cnts, _ = np.histogram(
                mc_in_time_cosmics_evt_df.loc[cos_mask, config.var_evt_reco_col],
                bins=config.bins,
                weights=mc_in_time_cosmics_evt_df.loc[cos_mask, WGT_COL]
            )
            del cos_mask, off_mask

            res_cos = get_covariance_matrix(np.array([cv_events + (d_cnts - m_cnts)]), cv_events)
            del d_cnts, m_cnts

            if PLOT_IND_UNCR:
                plot_frac_unc([(np.sqrt(np.diag(res_cos["cov_frac"])), "cosmic")], config)

            # C. Reweightables (Flux, GENIE, G4)
            reweight_covs = {}
            for s_name in ["Flux", "GENIE", "G4"]:
                print(f"--- {s_name} Syst ---")
                u_evts, cv_u = get_univ_rates(
                    cov_type="rate", evtdf=curr_mc, var_config=config,
                    syst_name=s_name, bkgd_subtract=False
                )
                reweight_covs[s_name] = get_covariance_matrix(u_evts, cv_u)
                del u_evts, cv_u
                if PLOT_IND_UNCR:
                    plot_frac_unc([(np.sqrt(np.diag(reweight_covs[s_name]["cov_frac"])), s_name)], config)

            # Sum totals in-place — no temporary matrices
            total_cov      += det_cov  # det uses frac as proxy for cov here
            total_cov_frac += det_cov_frac
            total_cov      += res_cos["cov"]
            total_cov_frac += res_cos["cov_frac"]
            for c in reweight_covs.values():
                total_cov      += c["cov"]
                total_cov_frac += c["cov_frac"]

            frac_unc_list += [
                (np.sqrt(np.diag(det_cov_frac)),        "Detector"),
                (np.sqrt(np.diag(res_cos["cov_frac"])), "Cosmic"),
            ]
            frac_unc_list += [(np.sqrt(np.diag(c["cov_frac"])), name) for name, c in reweight_covs.items()]
            del det_cov_frac, res_cos, reweight_covs

        # 4. Plotting & Saving
        final_plot_list = (
            [(np.sqrt(np.diag(total_cov_frac)), "Total")] + frac_unc_list
            if FULL_SYST else frac_unc_list
        )
        plot_frac_unc(final_plot_list, config)

        fig, p_value = plot_stacked_histogram_with_ratio(
            mc_df=curr_mc, data_df=curr_data, config=config,
            cov_frac_matrix=total_cov_frac, cov_matrix=total_cov,
            title=f"{cut} - {config.file_name}", weight_column=WGT_COL,
            data_pot=data_tot_pot, normalize=False,
            show_stats=FULL_SYST, divide_by_bin_width=divide_by_bin_width
        )
        del total_cov, total_cov_frac, curr_data, frac_unc_list

        save_dir = os.path.join(
            good_base_path if p_value > PVALUE_THRESHOLD else bad_base_path,
            config.file_name
        )
        os.makedirs(save_dir, exist_ok=True)
        f_name = f"cut_{cut}_{config.file_name}.pdf"
        fig.savefig(os.path.join(save_dir, f_name), format='pdf', bbox_inches='tight')
        plt.close(fig)
        del fig, curr_mc
        gc.collect()
        
        color = RED if p_value < PVALUE_THRESHOLD else ""
        print(f"{color}Saved {f_name} (Chi2: {p_value:.2f}){RESET}")

# --- 5. Archive Results ---
print(f"Archiving to {tar_output_path}...")
with tarfile.open(tar_output_path, "w:gz") as tar:
    tar.add(parent_path, arcname=os.path.basename(parent_path))
print("Done!")

# Proton Stuff

In [ ]:


import os, shutil, tarfile
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# --- Configuration & Setup ---


config_p_mu_np = FullHistogramConfig(
    file_name = "reco_p_mu",      
    var_evt_reco_col=('slc','measure_var','reco_p_mu','','',''),
    truth_column = ('truth', 'nu_categ_proton_reduced', '', '','',''),
    first_per_slice = True,
    start_cut = "energy",
    end_cut = "energy",
    bins=np.linspace(0,3, 41),
    xlabel=r'Muon candidate P [GeV]',
    ylabel=slices_y_label
)

config_p_mu_np_final = FullHistogramConfig(
    file_name = "reco_p_mu_final",      
    var_evt_reco_col=('slc','measure_var','reco_p_mu','','',''),
    truth_column = ('truth', 'nu_categ_proton_reduced', '', '','',''),
    first_per_slice = True,
    start_cut = "energy",
    end_cut = "energy",
    bins= np.array([0.1, 0.25, 0.4,0.6, 0.85, 3]),
    xlabel=r'Muon candidate P [GeV]',
    ylabel=slices_y_label
)

config_vec = [config_p_mu_np, config_p_mu_np_final]



FULL_SYST = True
if FULL_SYST:
    parent_path = "/exp/sbnd/data/users/lpelegri/Graphs/MCDataCompGraphsProtonFullErrNP"
else:
    parent_path = "/exp/sbnd/data/users/lpelegri/Graphs/MCDataCompGraphsProton"

if plot_sideband:
    parent_path += "sideband"
    
good_base_path, bad_base_path = os.path.join(parent_path, "good"), os.path.join(parent_path, "bad")
tar_output_path = f"{parent_path}.tar"

CHI2_THRESHOLD = 5
PLOT_IND_UNCR = False
PLOT_GENIE_CATEG = False
WGT_COL = ('slc', 'wgt', '', '', '', '')
SLICE_LEVELS = ['__ntuple', 'entry', 'rec.slc..index']

# Color codes
RED, RESET = "\033[31m", "\033[0m"

# Clean workspace
for path in [parent_path, tar_output_path]:
    if os.path.exists(path):
        shutil.rmtree(path) if os.path.isdir(path) else os.remove(path)
os.makedirs(good_base_path, exist_ok=True); os.makedirs(bad_base_path, exist_ok=True)

proton_mask_name_vec = ["0p","1p","2plusp"]

# Logic for the print statement
# --- 2. Main Loop ---




# --- Main Analysis Loop ---
for config in config_vec:
    for proton_mask_name in proton_mask_name_vec:
        if PLOT_GENIE_CATEG:
            config.truth_column = ('truth','genie_categ','','','','')
        else:
            config.truth_column = ('truth','nu_categ_proton_reduced','','','','')
                
        # Determine cut range
        idx_range = sorted([cuts.index(config.start_cut), cuts.index(config.end_cut)])
        selected_cuts = cuts[idx_range[0] : idx_range[1] + 1]
        
        for cut in selected_cuts: 
            print(f"--- Processing Cut: {cut} ---")
            
            # 1. Prepare Dataframes
            curr_mc = filter_df(mc_evt_df, config.extra_mask, mc_cumulative_masks[cut] & mask_dict[proton_mask_name](mc_evt_df), config)
            curr_data = filter_df(data_evt_df, config.extra_mask, data_cumulative_masks[cut]  & mask_dict[proton_mask_name](data_evt_df), config)
            
            # 2. Base Statistics (MC Stat Error)
            ret = signal_hists(evtdf=curr_mc, var_config=config, return_data=True, plot=False)
            cv_events = ret["nevts_allsel_reco"]
            ret_stats = get_stat_covariance_matrix(cv_events, ret["sum_w2_allsel_reco"])
            
            total_cov = ret_stats["cov"].copy()
            total_cov_frac = ret_stats["cov_frac"].copy()
            frac_unc_list = [(np.sqrt(np.diag(ret_stats["cov_frac"])), "MC Stat")]
        
            # 3. Systematic Blocks
            if FULL_SYST:
                # A. Detector Systematics
                print(f"--- Det Syst ---")
                syst_dfs = [filter_df(syst_evt_dfs[k], config.extra_mask, syst_masks[k][cut]  & mask_dict[proton_mask_name](syst_evt_dfs[k]), config) for k in syst_keys]
                n_det = detvar_plotter(evtdfs=syst_dfs, var_name=config.var_evt_reco_col, clip=config.clip, bins=config.bins)
                
                det_cov = np.zeros_like(total_cov)
                det_cov_frac = np.zeros_like(total_cov_frac)
                for i, key in enumerate(syst_keys):
                    if key == "SystVarsCV": continue
                    res = get_covariance_matrix(np.array([n_det[i]]), n_det[0])
                    det_cov += res["cov"]; det_cov_frac += res["cov_frac"]                        
                if PLOT_IND_UNCR:
                    plot_frac_unc([(np.sqrt(np.diag(det_cov_frac)), "total")], config)
        
                key_to_idx = {key: i for i, key in enumerate(syst_keys)}
                cv_events_det = n_det[0]
                ret_dict = {}
                for syst_key in unisim_keys:
                    if syst_key in key_to_idx:
                        kidx = key_to_idx[syst_key]
                        ret_dict[syst_key] = get_covariance_matrix(np.array([n_det[kidx]]), cv_events_det)
                for base_name, keys in paired_syst.items():
                    m_idx, p_idx = key_to_idx.get(keys[0]), key_to_idx.get(keys[1])
                    if m_idx is not None and p_idx is not None:
                        # Treats the +/- variations as universes of the same systematic
                        ret_dict[base_name] = get_covariance_matrix(np.array([n_det[m_idx], n_det[p_idx]]), cv_events_det)
        
                det_cov_frac = np.zeros_like(total_cov_frac)
                uncertanties_list = []
                for syst_key, ret in ret_dict.items():
                    frac_unc = np.sqrt(np.diag(ret["cov_frac"]))
                    uncertanties_list.append((frac_unc, syst_key))
                    det_cov_frac += ret["cov_frac"]
                    
                if PLOT_IND_UNCR:
                    plot_frac_unc(uncertanties_list + [(np.sqrt(np.diag(det_cov_frac)), "total")], config)
        
                # B. Cosmics
                print(f"--- Cosmics Syst ---")
                c_mc = filter_df(mc_in_time_cosmics_evt_df, config.extra_mask, in_time_cosmics_cumulative_masks[cut] & mask_dict[proton_mask_name](mc_in_time_cosmics_evt_df), config)
                c_data = filter_df(off_beam_light_evt_df, config.extra_mask, off_beam_data_cumulative_masks[cut]  & mask_dict[proton_mask_name](off_beam_light_evt_df), config)
                 
                d_cnts, _ = np.histogram(c_data[config.var_evt_reco_col], bins=config.bins, weights=c_data[WGT_COL])
                m_cnts, _ = np.histogram(c_mc[config.var_evt_reco_col], bins=config.bins, weights=c_mc[WGT_COL])
                res_cos = get_covariance_matrix(np.array([cv_events + (d_cnts - m_cnts)]), cv_events)    
                
                if PLOT_IND_UNCR:
                    plot_frac_unc([(np.sqrt(np.diag(res_cos["cov_frac"])), "cosmic")], config)
        
                # C. Reweightables (Flux, GENIE, G4)
                reweight_covs = {}
                for s_name in ["Flux", "GENIE", "G4"]:
                    print(f"--- {s_name} Syst ---")
                    u_evts, _ = get_univ_rates(cov_type="rate", evtdf=curr_mc, var_config=config, syst_name=s_name, bkgd_subtract=False)
                    reweight_covs[s_name] = get_covariance_matrix(u_evts, cv_events)
                    if PLOT_IND_UNCR:
                        plot_frac_unc([(np.sqrt(np.diag(reweight_covs[s_name]["cov_frac"])), s_name)], config)
        
                # --- Sum Totals ---
                total_cov += det_cov + res_cos["cov"] + sum(c["cov"] for c in reweight_covs.values())
                total_cov_frac += det_cov_frac + res_cos["cov_frac"] + sum(c["cov_frac"] for c in reweight_covs.values())
                
                # Append for final uncertainty plot
                frac_unc_list += [(np.sqrt(np.diag(det_cov_frac)), "Detector"), (np.sqrt(np.diag(res_cos["cov_frac"])), "Cosmic")]
                frac_unc_list += [(np.sqrt(np.diag(c["cov_frac"])), name) for name, c in reweight_covs.items()]
        
            
            # 4. Final Plotting & Saving
            if FULL_SYST:
                total_uncertainty_vec = np.sqrt(np.diag(total_cov_frac))
                final_plot_list = [(total_uncertainty_vec, "Total")] + frac_unc_list
            else:
                final_plot_list = frac_unc_list
            plot_frac_unc(final_plot_list, config)

            divide_bin_width = False
            if "final" in config.file_name:
                divide_bin_width = True
                
            fig, red_chi2 = plot_stacked_histogram_with_ratio(
                mc_df=curr_mc, data_df=curr_data, config=config,
                cov_frac_matrix=total_cov_frac, cov_matrix=total_cov,
                title=f"{cut} - {config.file_name}", weight_column=WGT_COL,
                data_pot=data_tot_pot, normalize=False, show_stats=FULL_SYST, divide_by_bin_width = divide_bin_width
            )
        
            # File Management
            status_path = good_base_path if red_chi2 < CHI2_THRESHOLD else bad_base_path
            save_dir = os.path.join(status_path, config.file_name)
            os.makedirs(save_dir, exist_ok=True)
            
            f_name = f"cut_{cut}_{config.file_name}_{proton_mask_name}.pdf"
            fig.savefig(os.path.join(save_dir, f_name), format='pdf', bbox_inches='tight')
            plt.close(fig)
        
            color = RED if red_chi2 > CHI2_THRESHOLD else ""
            print(f"{color}Saved {f_name} (Chi2: {red_chi2:.2f}){RESET}")

# --- 5. Archive Results ---
print(f"Archiving to {tar_output_path}...")
with tarfile.open(tar_output_path, "w:gz") as tar:
    tar.add(parent_path, arcname=os.path.basename(parent_path))
print("Done!")


In [ ]:
for column in mc_evt_df.truth:
    print(column)

# Proton Stuff

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch

# 1. Define Columns and Binning
score_col = ('slc','nu_score','','','','')
wgt_col   = ('slc', 'wgt', '', '', '', '')
bins = np.linspace(0, 1, 51)
bin_centers = (bins[:-1] + bins[1:]) / 2

# POT Scale (assuming this is defined in your environment)
# mc_pot_scale = ... 

for name, mask in data_cut_sequence:
    # --- Data Extraction ---
    # Filter and collapse to slice level (keeping the weight column)
    mc_nu_score_df = mc_evt_df[mc_cumulative_masks[name]].copy()
    data_nu_score_df = data_evt_df[data_cumulative_masks[name]].copy()
    
    slc_mc_df = mc_nu_score_df.groupby(['__ntuple', 'entry', 'rec.slc..index']).first()
    slc_data_df = data_nu_score_df.groupby(['__ntuple', 'entry', 'rec.slc..index']).first()
    
    # 2. Extract Values and apply POT weights to MC
    mc_vals = slc_mc_df[score_col].values
    mc_wgts = slc_mc_df[wgt_col].values  
    
    data_vals = slc_data_df[score_col].values
    data_wgts = slc_data_df[wgt_col].values
    
    # 3. Create the figure
    fig, ax1 = plt.subplots(figsize=(10, 7))

    # Vertical Cut Line at 0.55
    ax1.axvline(x=0.55, color='gray', linestyle='--', linewidth=2, label='Selection Cut (0.55)', zorder=2)
    
    # --- Plot the main Histograms (Weighted) ---
    mc_counts, _, _ = ax1.hist(mc_vals, bins=bins, weights=mc_wgts, 
                               histtype='stepfilled', color='gray', alpha=0.3, label='MC (Weighted)')
    
    data_counts, _ = np.histogram(data_vals, bins=bins, weights=data_wgts)
    data_counts_sq, _ = np.histogram(data_vals, bins=bins, weights=data_wgts**2)
    data_err = np.sqrt(data_counts_sq)
    
    ax1.errorbar(bin_centers, data_counts, yerr=data_err, fmt='ko', label='Data (Weighted)')
    
    ax1.set_xlabel("Nu Score", fontsize=14)
    ax1.set_ylabel("Weighted Events", fontsize=14)
    
    # --- Plot the Rejected Events Lines (Secondary Axis) ---
    ax2 = ax1.twinx()
    
    # Calculate cumulative sum of weights (rejection increases as threshold increases)
    mc_rejected_abs = np.cumsum(mc_counts)
    data_rejected_abs = np.cumsum(data_counts)
    
    mc_total = mc_rejected_abs[-1]
    data_total = data_rejected_abs[-1]
    
    mc_rejected_pct = (mc_rejected_abs / mc_total) * 100 if mc_total > 0 else np.zeros_like(mc_rejected_abs)
    data_rejected_pct = (data_rejected_abs / data_total) * 100 if data_total > 0 else np.zeros_like(data_rejected_abs)
    
    # --- Calculate Rejection Difference @ 0.55 ---
    # Find index of the bin center closest to 0.55
    cut_idx = np.argmin(np.abs(bin_centers - 0.55))
    mc_at_055 = mc_rejected_pct[cut_idx]
    data_at_055 = data_rejected_pct[cut_idx]
    diff_at_055 = data_at_055 - mc_at_055

    # Plot % Rejected
    ax2.plot(bin_centers, mc_rejected_pct, color='black', linestyle='--', linewidth=2, 
             label=f'MC % Rej ({mc_at_055:.1f}%: 0.55)')
    ax2.plot(bin_centers, data_rejected_pct, color='red', linestyle='--', linewidth=2, 
             label=f'Data % Rej ({data_at_055:.1f}%: 0.55)')
    
    # 4. Legend and Formatting
    ax2.set_ylabel("Percentage of Weighted Events Rejected (%)", color='red', fontsize=14)
    ax2.set_ylim(0, 105)
    ax2.tick_params(axis='y', labelcolor='red', labelsize=12)
    
    ax1.set_title(f"Stage: {name}", fontsize=16)
    
    # Combine legend handles
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    
    # Add a phantom patch for the Delta percentage
    diff_patch = Patch(color='none', label=f'Data-MC Diff: {diff_at_055:+.1f}%')
    
    all_lines = lines1 + lines2 + [diff_patch]
    all_labels = labels1 + labels2 + [f'Diff: 0.55: {diff_at_055:+.1f}%']
    
    ax1.legend(all_lines, all_labels, loc='upper left', fontsize=10, framealpha=0.9)
    
    plt.tight_layout()
    plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. Define Columns & Scan Range
score_col = ('slc','nu_score','','','','')
wgt_col   = ('slc', 'wgt', '', '', '', '')
scales    = np.linspace(0.9, 1.1, 101) 
bins      = np.linspace(0, 1, 41)
bin_centers = (bins[:-1] + bins[1:]) / 2

# Summary dictionary to track shifts across cuts
best_scales_summary = {}

for name, mask in data_cut_sequence:
    # --- YOUR REQUESTED DATA EXTRACTION ---
    # Filter and collapse to slice level (keeping the weight column)
    mc_nu_score_df = mc_evt_df[mc_cumulative_masks[name]].copy()
    data_nu_score_df = data_evt_df[data_cumulative_masks[name]].copy()
    
    slc_mc_df = mc_nu_score_df.groupby(['__ntuple', 'entry', 'rec.slc..index']).first()
    slc_data_df = data_nu_score_df.groupby(['__ntuple', 'entry', 'rec.slc..index']).first()
    
    # Extract Values and Weights
    mc_vals = slc_mc_df[score_col].values
    mc_wgts = slc_mc_df[wgt_col].values # Applying POT scale here
    
    data_vals = slc_data_df[score_col].values
    data_wgts = slc_data_df[wgt_col].values
    # ---------------------------------------

    # Pre-calculate Data histogram (constant for the inner scan)
    data_h, _ = np.histogram(data_vals, bins=bins, weights=data_wgts)
    data_h_sq, _ = np.histogram(data_vals, bins=bins, weights=data_wgts**2)

    # 3. Inner Loop: Scale Factor Scan
    chi2_results = []
    for s in scales:
        scaled_mc = mc_vals * s
        mc_h, _ = np.histogram(scaled_mc, bins=bins, weights=mc_wgts)
        mc_h_sq, _ = np.histogram(scaled_mc, bins=bins, weights=mc_wgts**2)
        
        # Variance denominator: Data Stat Error + MC Stat Error
        sigma2 = data_h_sq + mc_h_sq
        mask_sigma = sigma2 > 0
        
        # Pearson Chi2 calculation
        chi2 = np.sum(((data_h[mask_sigma] - mc_h[mask_sigma])**2) / sigma2[mask_sigma])
        chi2_results.append(chi2)

    # Find the best scale factor for this stage
    best_idx = np.argmin(chi2_results)
    best_scale = scales[best_idx]
    min_chi2 = chi2_results[best_idx]
    best_scales_summary[name] = best_scale

    # 4. Plotting
    fig, (ax_chi, ax_hist) = plt.subplots(1, 2, figsize=(16, 5))
    
    # Left: Chi2 Curve
    ax_chi.plot(scales, chi2_results, 'o-', color='tab:blue', markersize=4)
    ax_chi.axvline(best_scale, color='red', linestyle='--', label=f'Best Scale: {best_scale:.3f}')
    ax_chi.set_title(f"Stage: {name} - $\chi^2$ Profile", fontsize=14)
    ax_chi.set_xlabel("Scale Factor")
    ax_chi.set_ylabel(r"$\chi^2$")
    ax_chi.legend()

    # Right: Comparison Plot
    best_mc_vals = mc_vals * best_scale
    ax_hist.hist(best_mc_vals, bins=bins, weights=mc_wgts, 
                 histtype='stepfilled', color='gray', alpha=0.3, label=f'Best Fit (s={best_scale:.3f})')
    ax_hist.hist(mc_vals, bins=bins, weights=mc_wgts, 
                 histtype='step', color='blue', linestyle=':', label='Original MC (s=1.0)')
    ax_hist.errorbar(bin_centers, data_h, yerr=np.sqrt(data_h_sq), fmt='ko', label='Data')
    
    ax_hist.set_title(f"Stage: {name} - Comparison", fontsize=14)
    ax_hist.set_xlabel("Nu Score")
    ax_hist.set_ylabel("Weighted Events")
    ax_hist.legend(fontsize=9)
    
    plt.tight_layout()
    plt.show()

# Final Summary Printout
print("\n" + "="*40)
print(f"{'Cut Stage':<25} | {'Best Scale Factor':<15}")
print("-" * 40)
for stage, val in best_scales_summary.items():
    print(f"{stage:<25} | {val:.3f}")